Name: Nigus Bezabh Haile          Matricola: 947129  Email: n.haile@campus.unimib.it

# Implementation of a transformer layer (forward pass only) using NumPy.

Architecture:
1. Multi-head self-attention (8 heads) with residual connection.
2. Parallel feed-forward network (MLP with 1 hidden layer of 512 units) applied independently to each token, with residual connection.

Specifications:
- No PyTorch, only NumPy
- Forward pass only (no backprop)
- No batch size
- No LayerNorm (as specified)
- Random weights/biases
- Three input sequences: 10, 50, 100 tokens, each token a 256 embedding dimension

In [124]:
import numpy as np

# reproducibility
np.random.seed(42)

  ## Hyperparameters:

  `embedding_dim = 256`: embedding dimension (input/output token size)
  
  `num_heads = 8`: number of attention heads
  
  `dim_qkv = embedding dimension / number of attention heads = 32`: dimension of queries, keys, values per head (preserving the input                                                                          dimensionality and allowing the residual connection)
  
  `dim_feed_forward = 512`: hidden dimension of the parallel feed-forward network

In [125]:
embedding_dim = 256
num_heads = 8 
dim_qkv = embedding_dim // num_heads
dim_feed_forward = 512 

print(f"Embedding dimension (size of each input token:  {embedding_dim}")
print(f"Number of attention heads:     = {num_heads}")
print(f"Per head dimension of queries/keys/values = {dim_qkv}")
print(f"Hidden dimension of the parallel MLP  = {dim_feed_forward}")

Embedding dimension (size of each input token:  256
Number of attention heads:     = 8
Per head dimension of queries/keys/values = 32
Hidden dimension of the parallel MLP  = 512


## Input sequences

Three input sequences with respectively **10, 50, and 100 tokens**, each token represented by a **256-dimensional embedding**.
Each input is a matrix `X` of shape `(Embedding dimension, size of input tokens)`, where columns are tokens.

In [126]:
# Sequence lengths
sequence_lengths = [10, 50, 100]

# Generate three input matrices X of shape (Embedding dimension, size of input tokens). Each column is a token embedding of size = 256.
inputs = [np.random.randn(embedding_dim, token_size) for token_size in sequence_lengths]
for i, X in enumerate(inputs):
    print(f"Input sequence length {i+1}: (embedding_dim, token_size) = {X.shape}")

Input sequence length 1: (embedding_dim, token_size) = (256, 10)
Input sequence length 2: (embedding_dim, token_size) = (256, 50)
Input sequence length 3: (embedding_dim, token_size) = (256, 100)


## Helper: column wise softmax

Softmax applied independently on the columns of the attention score matrix $K^\top Q / \sqrt{D_q}$ (each column corresponds to one query token, and the column entries must sum to 1). Subtract the per column maximum for numerical stability.

In [127]:
def softmax_columns(M):
    M_shifted = M - np.max(M, axis=0, keepdims=True)  # numerical stability
    exp_M = np.exp(M_shifted)
    return exp_M / np.sum(exp_M, axis=0, keepdims=True)

## Single-head scaled dot-product self-attention

Given an input matrix $X \in \mathbb{R}^{D \times N}$, a single head computes

$$
V_h[X] = \beta_{v_h}\mathbf{1}^\top + \Omega_{v_h} X \qquad
Q_h[X] = \beta_{q_h}\mathbf{1}^\top + \Omega_{q_h} X \qquad
K_h[X] = \beta_{k_h}\mathbf{1}^\top + \Omega_{k_h} X
$$

$$
Sa_h[X] = V_h \cdot \mathrm{Softmax}\!\left[\frac{K_h^\top Q_h}{\sqrt{D_q}}\right]
$$

Each of $\Omega_{v_h}, \Omega_{q_h}, \Omega_{k_h}$ has shape `(dim_qkv, embedding_dim)` and each bias has shape `(dim_qkv, 1)`. The output of one head has shape `(dim_qkv, token_size)`.

In [128]:
def single_head_self_attention(X, omega_query, beta_query, Omega_key, beta_key, omega_value, beta_value):
    
    # Scaled dot-product self-attention for ONE head.

    # Parameters
    # X       : (embedding_dim, token_size) - input matrix (columns are token embeddings)
    # omega_query : (dim_qkv, embedding_dim) - query weight matrix
    # beta_query  : (dim_qkv, 1)             - query bias
    # Omega_key : (dim_qkv, embedding_dim)   - key weight matrix
    # beta_key  : (dim_qkv, 1)               - key bias
    # omega_value : (dim_qkv, embedding_dim) - value weight matrix
    # beta_value  : (dim_qkv, 1)             - value bias
    # Sa : (dim_qkv, token_size) - the output of this single attention head
    
    dimension_query = omega_query.shape[0]  # dimension of the queries (and keys)

    # Compute queries, keys, and values:
    Query = omega_query @ X + beta_query
    Key = Omega_key @ X + beta_key
    Value = omega_value @ X + beta_value

    # Attention scores
    scores = (Key.T @ Query) / np.sqrt(dimension_query)

    # Softmax over each column
    A = softmax_columns(scores)

    # Weighted sum of values
    Sa = Value @ A
    return Sa

## Multi-head self-attention

We run `num_heads = 8` heads in parallel, concatenate their outputs vertically, and project the result back to dimension `embedding_dim` with $\Omega_c$:

$$
\text{MhSa}[X] = \Omega_c \bigl[\,Sa_1[X]^\top,\, Sa_2[X]^\top,\, \dots,\, Sa_H[X]^\top\,\bigr]^\top
$$

Since each head outputs `(dim_qkv, token_size) = (embedding_dim/num_heads, token_size)`, the vertical concatenation gives a `(embedding_dim, token_size)` matrix, so $\Omega_c$ has shape `(embedding_dim, embedding_dim)` and the final output is `(embedding_dim, token_size)` — same shape as the input, which is exactly what we need for the residual connection.

In [129]:
def init_multi_head_params(embedding_dim, num_heads, dim_qkv):
    """Randomly initialize all multi-head attention parameters."""
    params = {
        'omega_query': [np.random.randn(dim_qkv, embedding_dim) * 0.1 for _ in range(num_heads)],
        'beta_query':  [np.random.randn(dim_qkv, 1) * 0.1 for _ in range(num_heads)],
        'omega_key': [np.random.randn(dim_qkv, embedding_dim) * 0.1 for _ in range(num_heads)],
        'beta_key':  [np.random.randn(dim_qkv, 1) * 0.1 for _ in range(num_heads)],
        'omega_value': [np.random.randn(dim_qkv, embedding_dim) * 0.1 for _ in range(num_heads)],
        'beta_value':  [np.random.randn(dim_qkv, 1) * 0.1 for _ in range(num_heads)],
        # Final linear projection that combines the heads back to dimension D.
        'Omega_c': np.random.randn(embedding_dim, num_heads * dim_qkv) * 0.1,
        'beta_c':  np.random.randn(embedding_dim, 1)         * 0.1,
    }
    return params


def multi_head_self_attention(X, params):
    
    num_heads = len(params['omega_query'])

    # Run each head independently
    head_outputs = []
    for h in range(num_heads):
        Sa_h = single_head_self_attention(
            X,
            params['omega_query'][h], params['beta_query'][h],
            params['omega_key'][h], params['beta_key'][h],
            params['omega_value'][h], params['beta_value'][h],
        )
        head_outputs.append(Sa_h)

    # Vertical (row-wise) concatenation
    concat = np.vstack(head_outputs)

    # Final linear projection
    out = params['Omega_c'] @ concat + params['beta_c']
    return out

## Parallel feed-forward network (MLP)

After multi-head attention (and its residual + skip), each token is processed **independently** by the same small MLP:

$$
\text{mlp}(x_n) = W_2 \, \mathrm{ReLU}(W_1 x_n + b_1) + b_2
$$

with one hidden layer of `dim_feed_forward = 512` units. "Independently" means we do **not** mix information across tokens here that mixing is the job of self-attention. Concretely, this is just a matrix multiplication on the columns of `X`, so when `X` has shape `(embedding_dim, token_size)`:

- $W_1$ has shape `(dim_feed_forward, embedding_dim)`, $b_1$ has shape `(dim_feed_forward, 1)`
- $W_2$ has shape `(embedding_dim, dim_feed_forward)`, $b_2$ has shape `(embedding_dim, 1)`

Output shape: `(embedding_dim, token_size)` — same as the input, ready for the second residual connection.

In [130]:
def relu(x):
    return np.maximum(0.0, x)


def init_mlp_params(embedding_dim, dim_feed_forward):
    """Randomly initialize parameters of the position-wise MLP."""
    return {
        'W1': np.random.randn(dim_feed_forward, embedding_dim)  * 0.1,
        'b1': np.random.randn(dim_feed_forward, 1)  * 0.1,
        'W2': np.random.randn(embedding_dim, dim_feed_forward)  * 0.1,
        'b2': np.random.randn(embedding_dim, 1)     * 0.1,
    }


def parallel_mlp(X, params):
    # Hidden layer
    H1 = relu(params['W1'] @ X + params['b1'])
    # Output layer
    out = params['W2'] @ H1 + params['b2']
    return out

## Full transformer layer

Putting it all together, with the two residual connections (and **no LayerNorm**):

$$
Z = X + \text{MhSa}(X)
$$
$$
\text{out} = Z + \text{mlp}(Z)
$$

The residual connections require that the shape `(embedding_dim, token_size)` is preserved everywhere, which is the reason we sized $\Omega_c$ to project the concatenated heads back to `embedding_dim`, and chose $W_2$ in the MLP to map back to `embedding_dim`.

In [131]:
def transformer_layer(X, attn_params, mlp_params):
    # First sub layer: multi-head self-attention + residual
    attn_out = multi_head_self_attention(X, attn_params)  
    Z = X + attn_out                                       

    # Second sublayer: parallel MLP + residual
    mlp_out = parallel_mlp(Z, mlp_params)           
    out = Z + mlp_out                                      

    return out

## Run it on the three input sequences

We initialize one set of weights (independent of `token_size`, which is the whole point of parameter sharing across positions in transformers) and apply the same transformer layer to all three sequences.

In [132]:
# Initialize the weights ONCE. The same set of parameters can be applied to
# sequences of any length token_size,this is the key property of transformers.
attn_params = init_multi_head_params(embedding_dim, num_heads, dim_qkv)
mlp_params  = init_mlp_params(embedding_dim, dim_feed_forward)

# Run the layer on each input sequence.
outputs = []
for i, X in enumerate(inputs):
    Y = transformer_layer(X, attn_params, mlp_params)
    outputs.append(Y)
    assert Y.shape == X.shape, "Output shape must match input shape for residuals to work."
    print(f"Input {i+1}: shape {X.shape}  ->  Output: shape {Y.shape}   "
          f"(token_size = {X.shape[1]}, mean = {Y.mean():+.4f}, std = {Y.std():.4f})")

Input 1: shape (256, 10)  ->  Output: shape (256, 10)   (token_size = 10, mean = -0.2902, std = 5.5810)
Input 2: shape (256, 50)  ->  Output: shape (256, 50)   (token_size = 50, mean = -0.1928, std = 4.7024)
Input 3: shape (256, 100)  ->  Output: shape (256, 100)   (token_size = 100, mean = -0.2197, std = 4.4707)


## Sanity checks

A few quick checks confirming that the implementation is consistent.

In [133]:
# Attention weights of each head form a valid probability distribution (columns of the softmax matrix are non-negative and sum to 1).
X_test = inputs[0]
Q = attn_params['omega_query'][0] @ X_test + attn_params['beta_query'][0]
K = attn_params['omega_key'][0] @ X_test + attn_params['beta_key'][0]
scores = (K.T @ Q) / np.sqrt(dim_qkv)
A = softmax_columns(scores)

print("Attention weights matrix A (single head, first input):")
print(f"  shape                     = {A.shape}")
print(f"  All entries >= 0          : {(A >= 0).all()}")
print(f"  Each column sums to 1     : {np.allclose(A.sum(axis=0), 1.0)}")
print(f"  Column sums (sample)      : {A.sum(axis=0)[:5]}")

Attention weights matrix A (single head, first input):
  shape                     = (10, 10)
  All entries >= 0          : True
  Each column sums to 1     : True
  Column sums (sample)      : [1. 1. 1. 1. 1.]


In [134]:
# the same set of parameters works for any sequence length.
for N_extra in [1, 5, 200, 500]:
    X_extra = np.random.randn(embedding_dim, N_extra)
    Y_extra = transformer_layer(X_extra, attn_params, mlp_params)
    print(f"token_size = {N_extra:4d}  ->  output shape {Y_extra.shape}  (OK)")

token_size =    1  ->  output shape (256, 1)  (OK)
token_size =    5  ->  output shape (256, 5)  (OK)
token_size =  200  ->  output shape (256, 200)  (OK)
token_size =  500  ->  output shape (256, 500)  (OK)


In [135]:
# Parameter count is independent of token_size (only depends on embedding dimension, number of heads, dim_feed_forward).
def count_params(attn_params, mlp_params):
    total = 0
    for key in ['omega_query', 'omega_key', 'omega_value', 'beta_query', 'beta_key', 'beta_value']:
        total += sum(M.size for M in attn_params[key])
    total += attn_params['Omega_c'].size + attn_params['beta_c'].size
    total += sum(M.size for M in mlp_params.values())
    return total

n_params = count_params(attn_params, mlp_params)
print(f"Total number of parameters in one transformer layer: {n_params:,}")
print(f"  -> independent of the sequence length (10, 50, 100, ...)")

Total number of parameters in one transformer layer: 526,080
  -> independent of the sequence length (10, 50, 100, ...)


In [136]:
# The parallel MLP is truly position-wise. Applying it on the full sequence column by column must give the same result

X_test = inputs[1]                              # (embedding_dim, 50)
Y_full = parallel_mlp(X_test, mlp_params)       # process all 50 tokens at once

# Process token by token.
Y_loop = np.zeros_like(Y_full)
for n in range(X_test.shape[1]):
    x_n = X_test[:, [n]]                        # (D, 1)
    Y_loop[:, [n]] = parallel_mlp(x_n, mlp_params)

print(f"Position-wise property (MLP is independent across tokens): "
      f"{np.allclose(Y_full, Y_loop)}")

Position-wise property (MLP is independent across tokens): True


In [137]:
# Stack several transformer layers (different weights for each layer) and confirm shapes are preserved end-to-end.
L = 4  # number of layers
layer_params = [
    (init_multi_head_params(embedding_dim, num_heads, dim_qkv), init_mlp_params(embedding_dim, dim_feed_forward))
    for _ in range(L)
]

for i, X in enumerate(inputs):
    h = X
    for attn_p, mlp_p in layer_params:
        h = transformer_layer(h, attn_p, mlp_p)
    print(f"Input {i+1} (token_size = {X.shape[1]:3d}) through {L} stacked transformer layers "
          f"-> output shape {h.shape}")

Input 1 (token_size =  10) through 4 stacked transformer layers -> output shape (256, 10)
Input 2 (token_size =  50) through 4 stacked transformer layers -> output shape (256, 50)
Input 3 (token_size = 100) through 4 stacked transformer layers -> output shape (256, 100)
